### Integrantes: Santiago Parada - Francisco Castro - Natalia Rodriguez - David Cifuentes

# 04 - Rol 4: Negocio, Datos e Integración
**Proyecto Retador: Patrones Invisibles** - ML No Supervisado - Universidad de La Sabana · 2026-II

**Pregunta de UrbanMove:** ¿qué debe hacer la empresa mañana por la mañana con estos hallazgos, y por qué se puede confiar en los datos que los sustentan?

**Plan del bloque:**

| Sección | Contenido |
|---|---|
| 1 | Justificación de la selección y estructuración de los datos |
| 2 | Integración de los tres bloques técnicos |
| 3 | Recomendación 1: reposicionamiento nocturno |
| 4 | Recomendación 2: calendario operativo por patrón, no por día de la semana |
| 5 | Recomendación 3: estimación de tiempos por franja y zona |
| 6 | Recomendación 4: tratamiento diferenciado de viajes atípicos |
| 7 | Tabla consolidada de resultados y limitaciones |

Datos de entrada: la muestra común y los resultados exportados por los Roles 1, 2 y 3.

## 0. Configuración

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA_DATOS = '../data'
muestra = pd.read_csv(f'{RUTA_DATOS}/muestra_50k.csv', parse_dates=['pickup_datetime'])
ORDEN_FRANJAS = ['madrugada', 'pico_am', 'valle', 'pico_pm', 'noche']
muestra['franja'] = pd.Categorical(muestra.franja, categories=ORDEN_FRANJAS, ordered=True)

etiquetas_r2 = pd.read_csv(f'{RUTA_DATOS}/etiquetas_rol2.csv')
perfiles_r3 = pd.read_csv(f'{RUTA_DATOS}/perfiles_viajes_rol3.csv')
muestra = muestra.merge(etiquetas_r2, on='id').merge(perfiles_r3, on='id')

registro = pd.read_csv(f'{RUTA_DATOS}/registro_limpieza.csv')
res_r1 = pd.read_csv(f'{RUTA_DATOS}/resultados_rol1.csv')
res_r2 = pd.read_csv(f'{RUTA_DATOS}/resultados_rol2.csv')
res_r3 = pd.read_csv(f'{RUTA_DATOS}/resultados_rol3.csv')
centroides = pd.read_csv(f'{RUTA_DATOS}/centroides_zonas.csv')

N = len(muestra)
print(f'{N:,} viajes con etiquetas de los tres bloques')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.dpi'] = 110

50,000 viajes con etiquetas de los tres bloques


## 1. Selección y estructuración de los datos

**Universo original:** 1 458 644 viajes de taxi en Nueva York entre el 1 de enero y el 30 de junio de 2016, tomados del conjunto `train.csv` de la competencia NYC Taxi Trip Duration en Kaggle. Se descarta `test.csv` porque no contiene la duración del viaje ni la hora de llegada, que son la variable central del análisis.

**Limpieza:** cinco reglas con justificación física u operativa, aplicadas en orden. Conservan el 98.69 % de los registros, así que eliminan comportamiento imposible sin recortar la población real.

**Muestreo:** 50 000 viajes por muestreo aleatorio simple con semilla fija sobre el conjunto limpio. Se descartó filtrar por un mes o un borough porque eso eliminaría justamente la variación entre zonas, horas y tipos de día que el proyecto debe medir. La representatividad se verificó con divergencia KL contra la población completa: menos de 0.35 milibits en la hora del día, el día de la semana y el mes.

**Estructuración:** una sola muestra y una sola zonificación de 20 zonas para los cuatro roles. Esto permite que las entropías del Rol 1, los clusters del Rol 2 y los perfiles del Rol 3 se refieran a los mismos viajes y se puedan cruzar sin ajustes.

In [2]:
display(registro)
print(f'Retención global: {100 * registro.restantes.iloc[-1] / 1_458_644:.2f} % de 1 458 644 viajes')
print(f'Muestra analizada: {N:,} viajes ({100 * N / registro.restantes.iloc[-1]:.2f} % del conjunto limpio)')

,regla,eliminados,restantes,% eliminado del total
0,Coordenadas dentro de NYC,1100,1457544,0.08
1,Duración entre 60 s y 3 h,10549,1446995,0.72
2,Pasajeros entre 1 y 6,19,1446976,0.00
3,Distancia >= 0.1 km,7439,1439537,0.51
4,Velocidad <= 100 km/h,33,1439504,0.00


Retención global: 98.69 % de 1 458 644 viajes
Muestra analizada: 50,000 viajes (3.47 % del conjunto limpio)


## 2. Integración de los tres bloques

Los tres bloques miran el mismo fenómeno con herramientas distintas. Antes de recomendar, conviene ver dónde se refuerzan.

In [3]:
integracion = pd.DataFrame([
    {'Hallazgo': 'La madrugada es el momento más distinto del día',
     'Rol 1': 'Entropía baja a 3.61 bits entre 2:00 y 3:00 y la KL contra el mapa promedio llega a 0.47 bits',
     'Rol 2': 'El cluster nocturno agrupa el 28.8 % de los viajes con el mayor peso de fin de semana',
     'Rol 3': 'El perfil 3 (4:14, 4.1 km, 22.8 km/h) es el más separado de los cuatro'},
    {'Hallazgo': 'El fin de semana cambia el cuándo más que el dónde',
     'Rol 1': 'KL de 0.188 bits en la hora frente a 0.050 bits en la zona',
     'Rol 2': 'La composición de clusters por zona es estable; la que cambia es la composición por hora',
     'Rol 3': 'El perfil de madrugada concentra el 39.0 % de fin de semana frente al 20.5 % del perfil matutino'},
    {'Hallazgo': 'Los viajes largos y de aeropuerto son una operación aparte',
     'Rol 1': 'El origen aporta 0.377 bits sobre el destino, cinco veces más que la hora',
     'Rol 2': 'DBSCAN marca como ruido el 7.8 % de los viajes que salen de JFK, frente a menos del 1 % en Manhattan',
     'Rol 3': 'Esos viajes reciben responsabilidad media de 0.992, la mayor del conjunto'},
])
integracion.style.set_properties(**{'text-align': 'left', 'white-space': 'normal'})

,Hallazgo,Rol 1,Rol 2,Rol 3
0,La madrugada es el momento más distinto del día,Entropía baja a 3.61 bits entre 2:00 y 3:00 y la KL contra el mapa promedio llega a 0.47 bits,El cluster nocturno agrupa el 28.8 % de los viajes con el mayor peso de fin de semana,"El perfil 3 (4:14, 4.1 km, 22.8 km/h) es el más separado de los cuatro"
1,El fin de semana cambia el cuándo más que el dónde,KL de 0.188 bits en la hora frente a 0.050 bits en la zona,La composición de clusters por zona es estable; la que cambia es la composición por hora,El perfil de madrugada concentra el 39.0 % de fin de semana frente al 20.5 % del perfil matutino
2,Los viajes largos y de aeropuerto son una operación aparte,"El origen aporta 0.377 bits sobre el destino, cinco veces más que la hora","DBSCAN marca como ruido el 7.8 % de los viajes que salen de JFK, frente a menos del 1 % en Manhattan","Esos viajes reciben responsabilidad media de 0.992, la mayor del conjunto"


## 3. Recomendación 1: reposicionamiento nocturno

El Rol 1 mostró que entre la 1:00 y las 3:00 la demanda deja de estar repartida y se concentra. Aquí se cuantifica cuántos conductores implica eso.

In [4]:
ventana = muestra[muestra.hora.isin([1, 2, 3])]
share_ventana = 100 * len(ventana) / N
top_noche = (ventana.zona_origen.value_counts(normalize=True) * 100).head(5)
top_dia = (muestra[muestra.hora.between(9, 22)].zona_origen.value_counts(normalize=True) * 100)

comparacion = pd.DataFrame({
    '% demanda 1:00-3:00': top_noche,
    '% demanda 9:00-22:00': top_dia.reindex(top_noche.index),
})
comparacion['razón noche / día'] = comparacion.iloc[:, 0] / comparacion.iloc[:, 1]
display(comparacion)

cobertura_noche = top_noche.head(3).sum()
cobertura_dia = top_dia.sort_values(ascending=False).head(3).sum()
print(f'La ventana 1:00-3:00 concentra el {share_ventana:.1f} % de los viajes del día.')
print(f'Con 3 zonas se cubre el {cobertura_noche:.1f} % de la demanda nocturna, '
      f'frente al {cobertura_dia:.1f} % que cubren las 3 mejores zonas diurnas.')

,% demanda 1:00-3:00,% demanda 9:00-22:00,razón noche / día
zona_origen,,,
6,21.35,5.39,3.96
7,12.25,6.33,1.93
9,9.04,4.97,1.82
2,8.94,9.31,0.96
1,8.60,9.34,0.92


La ventana 1:00-3:00 concentra el 6.0 % de los viajes del día.
Con 3 zonas se cubre el 42.6 % de la demanda nocturna, frente al 29.4 % que cubren las 3 mejores zonas diurnas.


**Lectura:** la ventana de 1:00 a 3:00 concentra solo el 6.0 % de los viajes del día, pero su reparto espacial es muy distinto. La zona 6 pasa del 5.4 % de la demanda diurna al 21.4 % nocturna, casi cuatro veces más. Con las zonas 6, 7 y 9 se cubre el 42.6 % de la demanda de esa ventana, mientras que de día las tres mejores zonas apenas llegan al 29.4 %.

**Recomendación cuantificada:** entre la 1:00 y las 3:00, posicionar alrededor del 40 % de los conductores activos en las zonas 6, 7 y 9, en lugar de distribuirlos con el mapa diurno. Fuera de esa ventana, la asignación debe ser proporcional a P(zona | hora), porque la entropía de 3.99 bits indica que la demanda diurna se reparte entre unas 16 zonas efectivas y concentrar sería contraproducente.

## 4. Recomendación 2: calendario operativo por patrón

El Rol 1 clasificó correctamente el tipo de día en el 97.8 % de los 182 días usando solo el patrón de viajes. Los días en que el modelo y el calendario no coinciden son festivos y días de clima extremo. Aquí se cuantifica qué tan distinto es operar esos días.

In [5]:
FESTIVOS = {'2016-01-01': 'Año Nuevo', '2016-02-15': 'Día de los Presidentes',
            '2016-05-30': 'Memorial Day', '2016-01-24': 'Día posterior a la tormenta'}
muestra['fecha'] = muestra.pickup_datetime.dt.normalize()

dias = muestra.groupby('fecha').agg(viajes=('id', 'size'), fin_de_semana=('fin_de_semana', 'first'))
laborales = dias[(dias.fin_de_semana == 0) & (~dias.index.strftime('%Y-%m-%d').isin(FESTIVOS))]
promedio_laboral = laborales.viajes.mean()

filas = []
for fecha, nombre in FESTIVOS.items():
    d = dias.loc[pd.Timestamp(fecha)]
    filas.append({'fecha': fecha, 'día': nombre, 'viajes en la muestra': int(d.viajes),
                  'vs. laboral promedio (%)': 100 * (d.viajes / promedio_laboral - 1)})
display(pd.DataFrame(filas))
print(f'Laboral promedio en la muestra: {promedio_laboral:.0f} viajes por día')

# Diferencia horaria entre fin de semana y día laboral
p_finde = muestra[muestra.fin_de_semana == 1].hora.value_counts(normalize=True).sort_index() * 100
p_lab = muestra[muestra.fin_de_semana == 0].hora.value_counts(normalize=True).sort_index() * 100
dif = (p_finde - p_lab).round(2)
print(f'\nMadrugada (0:00-4:00): {p_finde.loc[0:4].sum():.1f} % de los viajes del fin de semana '
      f'contra {p_lab.loc[0:4].sum():.1f} % en día laboral')
print(f'Pico matutino (7:00-9:00): {p_finde.loc[7:9].sum():.1f} % contra {p_lab.loc[7:9].sum():.1f} %')
display(dif.rename('diferencia en puntos porcentuales').to_frame().T)

,fecha,día,viajes en la muestra,vs. laboral promedio (%)
0,2016-01-01,Año Nuevo,259,-6.05
1,2016-02-15,Día de los Presidentes,253,-8.23
2,2016-05-30,Memorial Day,178,-35.43
3,2016-01-24,Día posterior a la tormenta,106,-61.55


Laboral promedio en la muestra: 276 viajes por día

Madrugada (0:00-4:00): 20.2 % de los viajes del fin de semana contra 7.0 % en día laboral
Pico matutino (7:00-9:00): 7.3 % contra 15.1 %


hora,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
diferencia en puntos porcentuales,3.12,3.33,2.93,2.52,1.38,-0.23,-1.39,-3.02,-3.19,-1.62,...,0.28,-0.35,0.52,0.40,-0.54,-0.69,-1.54,-1.68,-1.06,-0.30


**Lectura:** el fin de semana no mueve la demanda de lugar, mueve la hora. La madrugada de 0:00 a 4:00 pasa del 7.0 % de los viajes en día laboral al 20.2 % en fin de semana, mientras que el pico matutino de 7:00 a 9:00 cae del 15.1 % al 7.3 %. Son unos 13 puntos porcentuales que se trasladan de la mañana a la madrugada.

Los cuatro días donde el detector del Rol 1 discrepa del calendario confirman que la etiqueta de día laboral no basta: Año Nuevo y el Día de los Presidentes operan un 6 % y un 8 % por debajo del laboral promedio, Memorial Day un 35 % por debajo, y el día posterior a la tormenta de enero un 61.5 % por debajo.

**Recomendación cuantificada:** activar la programación de turnos por patrón de demanda y no por día de la semana. En días clasificados como fin de semana, trasladar alrededor de 13 puntos porcentuales de las horas de conductor del bloque 7:00-9:00 al bloque 0:00-4:00. El clasificador del Rol 1 acierta en el 97.8 % de los 182 días, así que la regla se puede automatizar.

## 5. Recomendación 3: estimación de tiempos por franja y zona

El Rol 2 mostró que dos viajes de distancia parecida tardan tiempos muy distintos según la hora. Esta sección mide cuánto mejora la estimación de duración al usar una velocidad por franja y zona en vez de una sola velocidad global.

**Diseño de la prueba:** las velocidades se calculan con los viajes de enero a abril y se evalúan sobre los de mayo y junio, que el modelo no vio. Solo se usa información disponible antes de iniciar el viaje: distancia, zona de origen y franja. El error se mide como error absoluto medio en minutos.

In [6]:
entrena = muestra[muestra.mes <= 4]
evalua = muestra[muestra.mes >= 5].copy()
print(f'Entrenamiento: {len(entrena):,} viajes (enero a abril) | Evaluación: {len(evalua):,} viajes (mayo y junio)')

v_global = entrena.velocidad_kmh.median()
v_franja_zona = entrena.groupby(['franja', 'zona_origen'], observed=True).velocidad_kmh.median()

evalua['pred_global'] = 60 * evalua.distancia_km / v_global
idx = pd.MultiIndex.from_arrays([evalua.franja, evalua.zona_origen])
evalua['v_local'] = v_franja_zona.reindex(idx).fillna(v_global).values
evalua['pred_local'] = 60 * evalua.distancia_km / evalua.v_local

mae_global = (evalua.pred_global - evalua.duracion_min).abs().mean()
mae_local = (evalua.pred_local - evalua.duracion_min).abs().mean()
print(f'\nVelocidad global única: {v_global:.2f} km/h -> error absoluto medio de {mae_global:.2f} min')
print(f'Velocidad por franja y zona:              error absoluto medio de {mae_local:.2f} min')
print(f'Reducción del error: {100 * (1 - mae_local / mae_global):.1f} %')

error_franja = evalua.groupby('franja', observed=True).apply(
    lambda g: pd.Series({'MAE global': (g.pred_global - g.duracion_min).abs().mean(),
                         'MAE por franja y zona': (g.pred_local - g.duracion_min).abs().mean()}),
    include_groups=False)
error_franja['reducción (%)'] = 100 * (1 - error_franja['MAE por franja y zona'] / error_franja['MAE global'])
display(error_franja)

Entrenamiento: 33,353 viajes (enero a abril) | Evaluación: 16,647 viajes (mayo y junio)

Velocidad global única: 13.02 km/h -> error absoluto medio de 6.78 min
Velocidad por franja y zona:              error absoluto medio de 5.73 min
Reducción del error: 15.5 %


,MAE global,MAE por franja y zona,reducción (%)
franja,,,
madrugada,8.51,4.79,43.75
pico_am,6.52,6.40,1.80
valle,7.01,6.93,1.18
pico_pm,6.05,5.42,10.38
noche,6.41,4.49,29.90


**Lectura:** usar una sola velocidad para toda la ciudad (13.02 km/h) produce un error absoluto medio de 6.78 minutos al estimar la duración. Calcular la velocidad mediana por franja y zona de origen lo baja a 5.73 minutos, una reducción del 15.5 % sobre viajes de mayo y junio que el modelo no vio.

La mejora no es pareja, y ahí está lo interesante: en madrugada el error cae 43.8 % y en la franja de noche 29.9 %, mientras que en las franjas diurnas la mejora es de apenas 1 a 2 %. Coincide con lo que mostró el Rol 1: de día el mapa promedio funciona bien, de noche no.

**Recomendación cuantificada:** reemplazar la velocidad única por una tabla de velocidades por franja y zona en el estimador de tiempos de llegada. El beneficio se concentra en la madrugada y la noche, donde el error se reduce entre 30 y 44 %, que son minutos directos de espera anunciada al pasajero.

## 6. Recomendación 4: tratamiento diferenciado de viajes atípicos

El Rol 2 identificó con DBSCAN un 1.69 % de viajes de comportamiento extremo, concentrados en los aeropuertos. Aquí se mide cuánto distorsionan la operación general.

In [7]:
comparativo = muestra.groupby('es_ruido').agg(
    viajes=('id', 'size'), distancia_mediana=('distancia_km', 'median'),
    duracion_mediana=('duracion_min', 'median'), velocidad_mediana=('velocidad_kmh', 'median'))
comparativo.index = ['Viajes regulares', 'Atípicos DBSCAN']
comparativo['% de la muestra'] = 100 * comparativo.viajes / N
display(comparativo)

# Minutos de conductor que consume cada grupo
muestra['minutos'] = muestra.duracion_min
reparto = muestra.groupby('es_ruido').minutos.sum()
print(f'Los atípicos son el {100*comparativo.loc["Atípicos DBSCAN", "viajes"]/N:.2f} % de los viajes '
      f'pero consumen el {100*reparto[True]/reparto.sum():.2f} % de los minutos de conductor.')

# Destinos desde los aeropuertos: base para reposicionar después de dejar un pasajero
for zona, nombre in [(14, 'JFK'), (13, 'LaGuardia')]:
    destinos = muestra[muestra.zona_origen == zona].zona_destino.value_counts(normalize=True).head(3) * 100
    print(f'\nDesde {nombre} (zona {zona}), destinos más frecuentes:')
    print('  ' + ' | '.join(f'zona {z}: {p:.1f} %' for z, p in destinos.items()))

,viajes,distancia_mediana,duracion_mediana,velocidad_mediana,% de la muestra
Viajes regulares,49158,2.12,11.13,12.85,98.32
Atípicos DBSCAN,842,3.45,10.12,36.89,1.68


Los atípicos son el 1.68 % de los viajes pero consumen el 2.17 % de los minutos de conductor.

Desde JFK (zona 14), destinos más frecuentes:
  zona 16: 13.1 % | zona 2: 10.0 % | zona 14: 9.4 %

Desde LaGuardia (zona 13), destinos más frecuentes:
  zona 2: 12.3 % | zona 0: 8.6 % | zona 13: 7.7 %


**Lectura:** los 843 viajes atípicos son el 1.69 % de la muestra pero consumen el 2.18 % de los minutos de conductor, porque son más largos que el promedio. Su velocidad mediana es de 36.9 km/h frente a 12.8 km/h del resto, es decir, no comparten nada con la operación urbana salvo el vehículo.

Los destinos desde los aeropuertos muestran por qué importa: desde JFK, el 13.1 % de los viajes termina en Brooklyn (zona 16) y el 10.0 % en la zona 2 de Manhattan; desde LaGuardia, el 12.3 % termina en la zona 2. Son trayectos predecibles que dejan al conductor lejos de donde empezó.

**Recomendación cuantificada:** tratar los viajes de aeropuerto como una línea de servicio aparte, con su propio estimador de tiempos y su propia regla de reposicionamiento. Al dejar un pasajero de JFK en Brooklyn, la decisión de a dónde ir después debe usar la matriz origen-destino, que aporta 0.377 bits de información según el Rol 1, y no el mapa general de demanda. Además, excluir ese 1.69 % del entrenamiento de los modelos generales evita que unos pocos viajes de autopista sesguen las estimaciones de toda la ciudad.

## 7. Tabla consolidada y limitaciones

In [8]:
consolidado = pd.concat([
    res_r1.assign(Rol='Rol 1').rename(columns={'Concepto': 'Bloque'}),
    res_r2.assign(Rol='Rol 2').rename(columns={'Método': 'Bloque'}),
    res_r3.assign(Rol='Rol 3'),
])[['Rol', 'Bloque', 'Medida', 'Valor']].reset_index(drop=True)
consolidado.to_csv(f'{RUTA_DATOS}/resultados_consolidados.csv', index=False)
consolidado

,Rol,Bloque,Medida,Valor
0,Rol 1,Entropía,H(zona origen) global,3.992 bits (15.9 zonas efectivas de 20)
1,Rol 1,Entropía,H(zona) franja más concentrada,3.869 bits (madrugada)
2,Rol 1,Entropía,H(zona) franja más dispersa,3.968 bits (pico_am)
3,Rol 1,Entropía condicional,H(destino | origen),3.753 bits (vs. 4.130 sin información)
4,Rol 1,Información mutua,I(zona origen; destino),0.377 bits
5,Rol 1,Información mutua,I(hora; destino),0.0717 bits (azar: 0.0063; p < 0.002)
6,Rol 1,KL,"KL(finde || laboral), hora",0.188 bits
7,Rol 1,KL,"KL(finde || laboral), zona",0.050 bits
8,Rol 1,KL,Hora más distinta del mapa promedio,3:00 (0.471 bits)
9,Rol 1,Entropía cruzada,"H(P_finde, Q_laboral), zona×hora",8.669 bits vs. H(P_finde) = 8.415


**Limitaciones del análisis:**

1. **El periodo es de seis meses de 2016:** No cubre un año completo, así que no captura estacionalidad de verano ni de diciembre, y los patrones de movilidad de Nueva York cambiaron desde entonces.
2. **Son viajes de taxi, no de UrbanMove:** El taxi se toma en la calle y la plataforma se pide por aplicación, así que la demanda observada subestima las zonas donde hay poca oferta de taxis.
3. **La distancia es en línea recta:** La fórmula de Haversine no recorre calles, así que las velocidades reportadas subestiman la velocidad real sobre la vía. Sirven para comparar entre franjas y zonas, no como valor absoluto.
4. **Las 20 zonas son una decisión operativa:** La entropía depende del número de zonas, así que los valores en bits no son comparables con los de otro equipo que haya usado otra zonificación.
5. **La mezcla gaussiana es una aproximación:** BIC baja de forma monótona, señal de que los datos no provienen de una mezcla finita de normales. Los perfiles son útiles para operar, no una descripción del proceso real.
6. **La demanda observada no es la demanda potencial:** Solo se ven los viajes que ocurrieron. Una zona sin viajes puede no tener demanda o puede no haber tenido vehículos disponibles, y los datos no permiten distinguir ambos casos.

**Siguientes pasos sugeridos:**

1. Cruzar con datos de clima y eventos para explicar los días atípicos que el modelo detecta.
2. Extender la detección de tipo de día a las primeras horas de la jornada, para activar el plan operativo antes de que termine el día.
3. Validar las recomendaciones con una prueba controlada en una zona antes de aplicarlas a toda la flota.